In [ ]:
# Install required packages
!pip install --upgrade --quiet ipywidgets pandas 'pydantic-ai-slim[duckduckgo,mcp,openai,openrouter,web-fetch]' python-dotenv braintrust requests tqdm

import os
import urllib.request
import zipfile

# Download and extract data files
url = 'https://github.com/jsoma/workshop-ai-agents/raw/main/docs/01-pydantic-ai-basics/01-pydantic-ai-basics-data.zip'
print(f'Downloading data from {url}...')
urllib.request.urlretrieve(url, '01-pydantic-ai-basics-data.zip')

print('Extracting 01-pydantic-ai-basics-data.zip...')
with zipfile.ZipFile('01-pydantic-ai-basics-data.zip', 'r') as zip_ref:
    zip_ref.extractall('.')

os.remove('01-pydantic-ai-basics-data.zip')
print('✓ Data files extracted!')

In [ ]:
import os
from getpass import getpass

from dotenv import load_dotenv

load_dotenv()

for key, label in [
    ("OPENROUTER_API_KEY", "OpenRouter API key"),
    ("OPENAI_API_KEY", "OpenAI API key, for the OpenAI web search examples"),
]:
    if not os.getenv(key):
        os.environ[key] = getpass(f"{label}: ")

# Pydantic AI basics

Pydantic AI - not to be confused with Pydantic! - is a library for interfacing with AI. It's not married to any individual provider (OpenAI, Anthropic, Google), so it's often more flexible and independent than other tools. The people who make it have a track record of quality involvement with the open-source ecosystem so I also trust its continued existence a lot more than other flashy startups.

We'll start by asking a **nice simple question** to an LLM.

In [2]:
from pydantic_ai import Agent

agent = Agent('openrouter:anthropic/claude-haiku-4-5')

result = await agent.run('Where does "hello world" come from?')  
print(result.output)

# Origins of "Hello World"

The first known "Hello World" program appeared in **1974** in a C tutorial by **Brian Kernighan**. It was a simple program demonstrating how to print text.

However, the phrase became widely popularized through **"The C Programming Language"** (1978), the famous textbook co-written by Kernighan and Dennis Ritchie. This book became the standard reference for C, and their "Hello World" example was adopted by countless programming tutorials that followed.

## Why "Hello World"?

There's no deep reason—it was simply a **convenient, recognizable phrase** for demonstrating output. The choice was somewhat arbitrary, though it's friendly and memorable. Some suggest it may have been influenced by an earlier computer science paper that used "hello, world" casually, but the exact inspiration isn't definitively documented.

## Why it stuck

Once it appeared in such an influential textbook, it became the **de facto first program** for learning any new language. The tradi

Instead of talking directly to Anthropic or OpenAI, we're using OpenRouter instead. OpenRouter offers a zillion and one models, along with much better API key management than dealing directly with the providers themselves. If you wanted to talk directly to openai, you definitely can - just use `openai:gpt-5-nano` instead.

In [3]:
from pydantic_ai import Agent

agent = Agent('openrouter:openai/gpt-5.4-nano')

result = await agent.run('Where does "hello world" come from?')  
print(result.output)

“Hello, world!” is a classic example used to teach basic programming concepts. It comes from programming culture rather than a single modern invention.

- **Origin (historical):** The phrase traces back to **the early days of the C language**. It was popularized by **Brian Kernighan and Dennis Ritchie** in **the book _The C Programming Language_ (1978)**, which famously used the “Hello, world” program as the first example.
- **Why it stuck:** It’s simple to type, easy to verify (“does it run?”), and demonstrates the basic idea of producing output without requiring much prior knowledge.
- **Earlier roots:** Variants of “hello world”-style messages existed even before C (in other languages and tutorials), but **Kernighan & Ritchie’s C book is what cemented it as the standard default example** for many languages.

So, while “hello world” may have appeared in earlier contexts, its widely recognized origin is **the C book (1978)** that made it the go-to beginner example.


In [4]:
from pydantic_ai import Agent

agent = Agent(
    'openrouter:openai/gpt-5.4-nano',
    instructions="Be very, very terse in your responses.")

result = await agent.run('Where does "hello world" come from?')  
print(result.output)

“Hello, world!” originated from **computer programming examples**—first popularized by **Brian Kernighan** in the early **1970s** (notably in *The C Programming Language*, 1978). It became a standard phrase to demonstrate that a language/toolchain is working.


# Structured output

One of the best use cases for AI is asking for **structured data** from something unstructured, like court cases. Maybe we have some text we extracted from a PDF of a lawsuit:

In [ ]:
lawsuit = """
Case No. 23STCV12345
Let it be known that a LAWSUIT has been filed in the Superior Court of California,
County of Los Angeles, on July 5 2028.

Barnaby Rutherford vs. Tamper Media LLC

on condition of fraud, breach of contract, and negligence.

The plaintiff alleges that the defendant failed to deliver the
agreed-upon services, resulting in financial losses and emotional distress.
The lawsuit seeks compensation for damages incurred and any additional relief
deemed appropriate by the court.
"""

A naive approach to extract from an LLM might look like the code below.

In [6]:
from pathlib import Path

from pydantic import BaseModel
from pydantic_ai import Agent, BinaryContent

MODEL = "openrouter:google/gemini-3.1-flash-lite"

prompt = """List the following about this lawsuit:
- case number
- court
- state
- filing date
- plaintiff
- defendant
- claims
"""

agent = Agent(MODEL)

result = await agent.run([prompt, lawsuit])

print(result.output)

Based on the information provided, here are the lawsuit details:

*   **Case Number:** 23STCV12345
*   **Court:** Superior Court of California, County of Los Angeles
*   **State:** California
*   **Filing Date:** July 5, 2028
*   **Plaintiff:** Jonathan Soma
*   **Defendant:** Tamper Media LLC
*   **Claims:** Fraud, breach of contract, and negligence


That's easier to read, but not perfect, though. We want something nice and programmatic, JSON or dictionaries! This is where **Pydantic** comes in. You build a model around what you want your response to look like.

In [7]:
from pathlib import Path

from pydantic import BaseModel
from pydantic_ai import Agent

MODEL = "openrouter:google/gemini-3.1-flash-lite"

class LawsuitInfo(BaseModel):
    case_number: str
    court: str
    state: str
    filing_date: str
    plaintiff: str
    defendant: str
    claims: list[str]

agent = Agent(MODEL,
              instructions="Extract the lawsuit information",
              output_type=LawsuitInfo)

result = await agent.run(lawsuit)

print(result.output)

case_number='23STCV12345' court='Superior Court of California, County of Los Angeles' state='California' filing_date='July 5 2028' plaintiff='Jonathan Soma' defendant='Tamper Media LLC' claims=['fraud', 'breach of contract', 'negligence']


## ...and it even works with images!

Take for example this car:

<img src="data/car.jpg">

Just like the lawsuit, we can feed it directly to an LLM and ask questions about it.

In [8]:
from pathlib import Path
from pydantic import BaseModel
from pydantic_ai import Agent, BinaryContent

MODEL = "openrouter:google/gemini-3.1-flash-lite"
DATA = Path("data")

class VehicleInfo(BaseModel):
    make: str | None
    model: str | None
    type: str | None
    color: str | None
    license_plate: str | None
    estimated_year: str | None

image = BinaryContent(
    data=(DATA / "car.jpg").read_bytes(),
    media_type="image/jpeg",
)

agent = Agent(MODEL, 
              instructions="Extract the appropriate vehicle information",
              output_type=VehicleInfo)

result = await agent.run([prompt, image])

print(result.output)

make='Toyota' model='Highlander' type='SUV' color='beige' license_plate='2C-6784' estimated_year='2004-2007'


# Working with a lot of inputs

Pydantic and structured outputs shine when you have a *lot* of data, like all of these car photos.

<img src="data/cars.png">

You make the same setup as before.

> While we're at it I'm also going to get *very detailed* about what we're asking for. We could have done this before but I was trying to keep things simple!

In [9]:
from pathlib import Path

import pandas as pd
from pydantic import BaseModel, Field
from pydantic_ai import Agent, BinaryContent
from typing import Literal

MODEL = "openrouter:google/gemini-3.1-flash-lite"
DATA = Path("data")

class Vehicle(BaseModel):
    make: str = Field(description="Vehicle manufacturer")
    model: str = Field(description="Vehicle model name")
    color: str = Field(description="Primary color")
    year_estimate: int = Field(description="Estimated model year")
    vehicle_type: Literal[
        "sedan", "SUV", "truck", "van", "motorcycle", "other"
    ] = Field(description="Type of vehicle")
    confidence: float = Field(description="Confidence in identification, 0.0 to 1.0")
    license_plate: str | None

agent = Agent(MODEL, output_type=Vehicle)

...and then you just loop through it, collecting the outputs and pushing them into a dataframe.

In [10]:
from tqdm import tqdm

PROMPT = "Analyze the vehicle in this image. Fill in all fields."

rows = []
image_paths = sorted((DATA / "cars").glob("*.jpg"))
for image_path in tqdm(image_paths):
    # Get the result
    image = BinaryContent(data=image_path.read_bytes(), media_type="image/jpeg")
    result = await agent.run([PROMPT, image])

    # Save the result
    row = result.output.model_dump()
    row["filename"] = image_path.name
    rows.append(row)

print(f"Processed {len(rows)} images.")

5it [00:13,  2.62s/it]

Processed 5 images.


In [11]:
df = pd.DataFrame(rows)
df


,make,model,color,year_estimate,vehicle_type,confidence,license_plate,filename
0,Toyota,Yaris,yellow,2018,sedan,1.0,7กถ 4059,28262480.jpg
1,Toyota,Camry,silver,2014,sedan,1.0,0050 60A,28246768.jpg
2,Volkswagen,Multivan (T6),black,2018,van,1.0,K02VV,28246634.jpg
3,Tesla,Model Y,white,2022,SUV,1.0,沪A AB2295,28266737.jpg
4,Lexus,LX 570,black,2018,SUV,1.0,ZJA 777,28262472.jpg


# Adding tools

Talking to an LLM one step at a time is *fine*, but that isn't what makes something **agentic**.

Agentic work is about giving the LLM options and letting it work independently until it decides it has come to an answer.

We'll start with **WebSearch**, which... searches the web.

In [ ]:
from pydantic_ai import Agent
from pydantic_ai.capabilities import WebSearch, WebFetch

MODEL = 'openrouter:anthropic/claude-haiku-4-5'

# Uses the web search built-in to the LLM provider
agent = Agent(
    MODEL,
    capabilities=[WebSearch(local=False)],
)

prompt = """
Research who Jonathan Soma is and provide a two-sentence summary
of who he likely is.
"""

result = await agent.run(prompt)
print(result.output)

Based on the search results, Jonathan Soma is a data journalism educator and instructor at [Columbia Journalism School](https://journalism.columbia.edu/faculty/jonathan-soma), where he teaches courses like Data Journalism and Data Analysis Studio. He also runs a cat rescue and maintains an active presence in the tech community as a developer and educator, as evidenced by his [GitHub profile](https://github.com/jsoma) with 278 public repositories.


You can provide all sorts of options to WebSearch, the most useful are probably site-specific search and location-specific search.

> We'll talk about this more, but WebSearch by default is a **provider tool**, not something you can infinitely customize and have control over. It runs on *OpenAI's servers* or *Anthropic's servers* (or whoever else's), and most of the customization is only available in this "native" format, not with the "local" (on your computer) approach.

In [ ]:
from pydantic_ai import Agent, WebSearchTool, WebSearchUserLocation
from pydantic_ai.capabilities import NativeTool

# Different LLM providers have different options
# allowed_domains= does not work with openrouter!

agent = Agent(
    # 'openrouter:anthropic/claude-haiku-4-5',
    'openai-responses:gpt-5.4',
    capabilities=[
        NativeTool(
            WebSearchTool(
                search_context_size='medium',
                user_location=WebSearchUserLocation(
                    city='New York',
                    country='US',
                    region='NY'
                ),
                allowed_domains=['brooklynbrainery.com'],
            )
        )
    ],
)

result = await agent.run('Research who Jonathan Soma is and provide a two-sentence summary of who he likely is.')
print(result.output)

Based on the search results, Jonathan Soma is a data journalism educator and instructor at [Columbia Journalism School](https://journalism.columbia.edu/faculty/jonathan-soma) who teaches courses on data journalism, data analysis, and storytelling with data. He also runs a cat rescue and maintains an active presence in the data journalism community, as evidenced by his work with organizations like the Lede Program and his [GitHub profile](https://github.com/jsoma) with numerous public repositories.


# Run your tools locally

Search: Perplexity, Exa, Tavily, DDG

In [16]:
from pydantic_ai import Agent
from pydantic_ai.common_tools.duckduckgo import duckduckgo_search_tool

agent = Agent(
    'openrouter:anthropic/claude-haiku-4-5',
    tools=[
        duckduckgo_search_tool(max_results=10)
    ]
)

result = await agent.run("""
Research who Jonathan Soma is and provide a two-sentence summary of who he likely is and what he's teaching this semester.
""")
print(result.output)

Based on my research, here's a two-sentence summary of Jonathan Soma:

Jonathan Soma is the Knight Chair in Data Journalism at Columbia University's Journalism School, where he directs the Data Journalism MS program and the Lede Program summer intensive. This semester, he is likely teaching courses in data journalism, Python programming, data analysis, and data visualization—topics he specializes in making accessible to journalists and students.


## Custom tools

So far we've only seen tools that search the web. Maybe you have another source of information: an API, local documents, a Slack channel, etc etc etc.

Custom tools allow you to enable the agent to do things it can't do out-of-the-box. Acquiring information is just one tiny slice of opportunity! 

In [18]:
import requests
from datetime import datetime

# The docstring explains to the agent what the tool does

def get_date() -> str:
    """Get the current date."""
    return datetime.now().strftime("%Y-%m-%d")

def get_jrn_schedule(semester: Literal["Fall", "Spring", "Summer"], year: int) -> str:
    """Get Columbia Journalism School's schedule details for a given semester, and year."""

    # https://doc.sis.columbia.edu/sel/JOUR_Spring2026_text.html
    response = requests.get(f"https://doc.sis.columbia.edu/sel/JOUR_{semester}{year}_text.html")
    if response.status_code != 200:
        return "Schedule not found."
    return response.text

In [22]:
text = get_jrn_schedule("Fall", 2023)
text[:2000]

'<!DOCTYPE html>\n<html><head>\n<title>Department Listing: Journalism Courses in the Fall 2023 Semester</title>\n<meta charset="utf-8">\n<meta name="robots" content="noindex,nofollow">\n<link rel="stylesheet" href="../doc_main.v02.css">\n</head>\n<body><div id="text-header">\n<h1>Department Listing: Journalism Courses in the Fall 2023 Semester</h1></div>\n<pre>\n     Number Sec  Call#      Pts  Title                           Day Time          Room Building        Faculty\n                 L App Activity  Subject\n\nREGI J0001  001  12878     1-12  REGISTERED FOR JOURNALISM                                              \n                       DUMMY CO  Registered                                                             \nRESI J0001  001  12879        0  1-RESIDENCE UNIT F-T JOUR                                              \n                       INDEPEND  Residence Unit                                                         \nRSRH J0001  001  12880        0  RESEARCH-JOURNALISM  

Let's give it a try: this agent has a handful of tools. See what happens if you add or remove them!

In [25]:
from pydantic_ai import Agent
from pydantic_ai.common_tools.duckduckgo import duckduckgo_search_tool

agent = Agent(
    'openrouter:anthropic/claude-haiku-4-5',
    tools=[
        duckduckgo_search_tool(max_results=10),
        # get_date,
        get_jrn_schedule
    ]
)

result = await agent.run("""
    Research who Jonathan Soma is and provide a two-sentence summary of who he is and what he's teaching this semester.
""")
print(result.output)

Based on my research, here is a two-sentence summary of Jonathan Soma:

Jonathan Soma is the Knight Chair in Data Journalism at Columbia Journalism School, where he directs both the Data Journalism Master's program and the summer intensive Lede Program, and is known for his work making unapproachable data accessible to journalists. This fall semester (Fall 2024), he is teaching "Foundations of Computing" (JOUR J4001) on Mondays and Wednesdays from 9:00 AM to 12:00 PM.


### Seeing what tools were called, and with what

You *can* look under the hood to see what tools were called, but it isn't very attractive.

In [82]:
from pydantic_ai import ToolCallPart

for message in result.all_messages():
    for part in message.parts:
        if isinstance(part, ToolCallPart):
            print("\n## TOOL CALLED")
            print("name:", part.tool_name)
            print("args:", part.args_as_dict())


## TOOL CALLED
name: duckduckgo_search
args: {'query': 'Jonathan Soma'}

## TOOL CALLED
name: get_jrn_schedule
args: {'semester': 'Fall', 'year': 2024}


To do this "properly" we're going to **instrument our agent**.

# Adding instrumentation

**Observability** is the ability to see what's going on inside of your agent. **Instrumentation** is the process of adding the tools to your agent that allow it to be observed.

We're going to use [Braintrust](https://www.braintrust.dev/), but [LogFire](https://pydantic.dev/logfire), [LangFuse](https://langfuse.com/), [Arize Phoenix](https://arize.com/phoenix/) are all other alternatives. The space is crowded!

In [27]:
from braintrust.wrappers.pydantic_ai import setup_pydantic_ai

setup_pydantic_ai(project_name="dataharvest-2026")

True

In [28]:
from pydantic_ai import Agent
from pydantic_ai.common_tools.duckduckgo import duckduckgo_search_tool

agent = Agent(
    'openrouter:anthropic/claude-haiku-4-5',
    tools=[
        duckduckgo_search_tool(max_results=10),
        get_date,
        get_jrn_schedule
    ]
)

result = await agent.run("""
    Research who Jonathan Soma is and provide a two-sentence summary of who he is and what he taught last semester.
""")
print(result.output)

Perfect! I found the information I needed. Based on my research, here's a two-sentence summary:

**Jonathan Soma is the Knight Chair in Data Journalism at Columbia Journalism School, directing both the Data Journalism MS program and serving as an educator focused on making data accessible and approachable.** Last semester (Spring 2026), he taught "Data, Computation, Innovation," a lecture course in the journalism department.


Now I can go see what happened in my [Braintrust logs](https://www.braintrust.dev/).

As a point of comparison, try it with the following:

In [ ]:
from pydantic_ai import Agent

agent = Agent(
    'openrouter:anthropic/claude-haiku-4-5',
    capabilities=[WebSearch(), WebFetch()]
)

result = await agent.run("""
    Research who Jonathan Soma is and provide a two-sentence summary of who he is and what he taught last semester.
""")
print(result.output)

/var/folders/25/h3prywj14qb0mlkl2s8bxq5m0000gn/T/ipykernel_92579/187993457.py:5: PydanticAIDeprecationWarning: WebSearch will stop auto-selecting DuckDuckGo based on package availability in v2. To keep this fallback, pass `local='duckduckgo'` (or `local=True`). To disable the fallback, pass `local=False`.
  capabilities=[WebSearch()]


# Jonathan Soma

[jonathansoma.com](https://jonathansoma.com/) indicates that Jonathan Soma is the Knight Chair of Data Journalism at Columbia University's Journalism School, where he runs a cat rescue, teaches data journalism, and creates educational resources about Python, AI, and data visualization. Based on the [Columbia University course catalog](https://peqod.com/prof/Jonathan_Soma), last semester (Fall 2025) he taught JOUR J4001, a data journalism course that met on Mondays and Wednesdays.


# Capabilities and customization

Pydantic can be confusing - tools, toolsets, capabilities, MCP servers... and it doesn't even stay stable! We accept Pydantic's faults, though, and move on.

## Capabilities

Pydantic has *some* built-in abilities, but they recently moved some of them into the [Pydantic AI harness](https://github.com/pydantic/pydantic-ai-harness) and let random folks take over some of the implementations and it's just... a little messy at the moment. Take a breath.

- Capabilities: https://pydantic.dev/docs/ai/core-concepts/capabilities
- Some common ones: https://github.com/vstorm-co/pydantic-ai-backend

vstorm seems to have done a lot with Pydantic *but* that repo has under a hundred stars, so I don't feel great about it. This is why below we're using the official modelcontextprotocol filesystem MCP server instead of the Pydantic AI Backend one, even though the backend one claims to sandbox etc.

## MCP Servers

MCP servers are your ability to talk to some other piece of software. Bridges or APIs, if you will.

Below we use one for the [filesystem](https://github.com/modelcontextprotocol/servers/tree/main/src/filesystem) and one for [Powerpoint](https://github.com/GongRzhe/Office-PowerPoint-MCP-Server).

> Point of discussion: why an MCP Server for Powerpoint instead of a Powerpoint library that runs in Python?

In [89]:
from pydantic_ai.mcp import MCPToolset

In [ ]:
file_server = MCPToolset({
    "mcpServers": {
        "filesystem": {
            "command": "npx",
            "args": [
                "-y",
                "@modelcontextprotocol/server-filesystem",
                str("."),
            ],
        }
    }
})

ppt_server = MCPToolset({
  "mcpServers": {
    "ppt": {
      "command": "uvx",
      "args": [
        "--from", "office-powerpoint-mcp-server", "ppt_mcp_server"
      ],
      "env": {}
    }
  }
})

# We'll start without toolsets
agent = Agent(
    'openrouter:anthropic/claude-haiku-4-5',
    # toolsets=[file_server, ppt_server],
)

instructions = f"""
You are a helpful assistant.
"""

prompt = f"""
What files are in the current directory? Make a powerpoint presentation about how you did it.
"""

result = await agent.run(prompt,
                        instructions=instructions)

print(result.output)

Perfect! Here's what I found in the current directory:

## Files in the Directory:

**Files:**
- `.DS_Store` - macOS system file
- `Pydantic AI basics.ipynb` - Jupyter notebook for learning Pydantic AI
- `public-records-story-candidates.md` - Markdown document
- `talk.md` - Markdown document

**Directories:**
- `data/` - Directory for data files
- `outputs/` - Directory for output files

## How I Did It:

I created a 5-slide PowerPoint presentation that documents the process:

1. **Title Slide** - Introduction to the process
2. **Step 1** - Explained how I used `list_allowed_directories()` to find the authorized directory path
3. **Step 2** - Showed how I used `list_directory()` to scan the folder structure
4. **Files Found** - Listed all 6 items (4 files, 2 directories) with descriptions
5. **Summary** - Outlined the complete technical process

The presentation has been saved as `Directory_Listing_Process.pptx` in your current directory!
